In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

In [2]:
# b. Upload / Access the dataset (MNIST digit 0 treated as normal data, rest as anomalies)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape to (N, 784)
x_train = x_train.reshape((len(x_train), -1))
x_test = x_test.reshape((len(x_test), -1))

# Train ONLY on Normal Class (e.g., Digit '0')
normal_mask = (y_train == 0)
x_train_normal = x_train[normal_mask]

# Create Test Set with both normal ('0') and anomalous (all other digits) data
x_test_normal = x_test[y_test == 0]
x_test_anomaly = x_test[y_test != 0][:1000] # Pick 1000 anomalies for evaluation

In [3]:
# c. Encoder & d. Decoder networks
class Autoencoder(Model):
    def __init__(self, latent_dim=32):
        super(Autoencoder, self).__init__()
        # c. Encoder converts input into compressed latent representation
        self.encoder = tf.keras.Sequential([
            layers.Dense(128, activation='relu'),
            layers.Dense(latent_dim, activation='relu')
        ])
        # d. Decoder networks convert latent representation back to original input dimension
        self.decoder = tf.keras.Sequential([
            layers.Dense(128, activation='relu'),
            layers.Dense(784, activation='sigmoid')
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [4]:
# Initialize Autoencoder
autoencoder = Autoencoder(latent_dim=32)

# e. Compile model with Optimizer, Loss, and Evaluation Metrics
autoencoder.compile(optimizer='adam', loss='mae', metrics=['mse'])

# Train model exclusively on normal data
autoencoder.fit(x_train_normal, x_train_normal,
                epochs=10,
                batch_size=256,
                shuffle=True,
                validation_data=(x_test_normal, x_test_normal),
                verbose=1)

# Detect Anomalies by Reconstruction Loss
reconstructions_normal = autoencoder.predict(x_test_normal)
loss_normal = np.mean(np.abs(reconstructions_normal - x_test_normal), axis=1)

reconstructions_anomaly = autoencoder.predict(x_test_anomaly)
loss_anomaly = np.mean(np.abs(reconstructions_anomaly - x_test_anomaly), axis=1)

# Set Threshold at 95th percentile of normal reconstruction loss
threshold = np.percentile(loss_normal, 95)
print(f"\nCalculated Anomaly Threshold: {threshold:.4f}")

# Anomaly Detection Results
normal_correct = np.sum(loss_normal < threshold) / len(loss_normal)
anomalies_detected = np.sum(loss_anomaly > threshold) / len(loss_anomaly)

print(f"Normal Samples Correctly Classified: {normal_correct * 100:.2f}%")
print(f"Anomalous Samples Detected: {anomalies_detected * 100:.2f}%")

Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 0.2999 - mse: 0.1434 - val_loss: 0.1254 - val_mse: 0.0898
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.1233 - mse: 0.0847 - val_loss: 0.1175 - val_mse: 0.0774
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.1130 - mse: 0.0751 - val_loss: 0.1074 - val_mse: 0.0729
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.1049 - mse: 0.0718 - val_loss: 0.1002 - val_mse: 0.0687
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0992 - mse: 0.0685 - val_loss: 0.0964 - val_mse: 0.0669
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0967 - mse: 0.0675 - val_loss: 0.0952 - val_mse: 0.0664
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0957 - mse: 0.0669 - val_loss: 0.0940 - val_mse: 0.0655
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0942 - mse: 0.0655 - val_loss: 0.0922 - val_mse: 0.0638
Epoch 9/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.091